In [6]:
import pandas as pd 
import numpy as np

Timestamp: request submission time, seconds from 0:00:00 on the first day (calibrated to the local time zone ).

Session ID: conversation ID, only conversation mode have this, traces that share the same value of Session ID are in the same conversation session.

Elapsed time: time between the request submission time and system response time, covers the full duration from submitting the inference request to getting the complete final response (all tokens included). It’s not the same as TTFT (time to first token). In seconds.

Model: called models, including ChatGPT(GPT-3.5) and GPT-4.

Request tokens: Request tokens length.

Response tokens: Response tokens length.

Total tokens: Request tokens length plus response tokens length.

Log Type: the way users call the model, in conversation mode or using API, including Conversation log and API log.

In [ ]:
INPUT_FILE  = '/Users/ritaaf/Desktop/inference/BurstGPT_without_fails_3.csv'  
OUTPUT_FILE = 'arrivals.csv'
df = pd.read_csv(INPUT_FILE, usecols=['Timestamp', 'Elapsed time'])

Loading /Users/ritaaf/Desktop/inference/BurstGPT_without_fails_3.csv ...
  Loaded: 4,956,058 rows


In [ ]:
# Drop any rows with missing values
df = df.dropna()

# Drop rows where elapsed time is 0 or negative (failed requests)
df = df[df['Elapsed time'] > 0]

After cleaning: 2,161,220 rows


In [9]:
# Timestamp is in seconds -> divide by 3600 to get hours
df['arrival_timestamp'] = df['Timestamp'] / 3600.0
df['arrival_timestamp'] -= df['arrival_timestamp'].min()

# Elapsed time is in seconds -> divide by 3600 to get hours
df['service_time'] = df['Elapsed time'] / 3600.0

# Sort chronologically
df = df.sort_values('arrival_timestamp').reset_index(drop=True)


t_start = df['arrival_timestamp'].min()
t_end   = df['arrival_timestamp'].max()
print(f"\n  Time span : {t_start:.1f} hrs → {t_end:.1f} hrs")
print(f"  Duration  : {(t_end - t_start):.1f} hrs  "
      f"({(t_end - t_start)/24:.1f} days)")

print(f"\n  Elapsed time stats (seconds):")
et = df['service_time'] * 3600
print(f"    mean   : {et.mean():.2f}s")
print(f"    median : {et.median():.2f}s")
print(f"    max    : {et.max():.2f}s")
print(f"    >300s  : {(et > 300).sum():,} rows  (possible outliers)")

# Per-hour arrival rate check
df['hour_bin'] = df['arrival_timestamp'].astype(int)
hourly = df.groupby('hour_bin').size()
print(f"\n  Arrivals/hr — mean: {hourly.mean():.0f}  "
      f"max: {hourly.max():.0f}  "
      f"min: {hourly.min():.0f}")



  Time span : 0.0 hrs → 2640.0 hrs
  Duration  : 2640.0 hrs  (110.0 days)

  Elapsed time stats (seconds):
    mean   : 6.10s
    median : 2.00s
    max    : 714.00s
    >300s  : 43 rows  (possible outliers)

  Arrivals/hr — mean: 836  max: 15480  min: 1


In [10]:

out = df[['arrival_timestamp', 'service_time']]
out.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(out):,} rows → {OUTPUT_FILE}")
print("\nFirst 5 rows:")
print(out.head().to_string(index=False))

Saved 2,161,220 rows → arrivals.csv

First 5 rows:
 arrival_timestamp  service_time
          0.000000      0.011944
          0.014167      0.000556
          0.022778      0.002222
          0.040000      0.000833
          0.053056      0.007222
